<a href="https://colab.research.google.com/github/yosepdoni/Advanced-Artificial-Intelligence/blob/main/RawNet3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Sel 1: Install semua dependency
!pip install -q speechbrain==0.5.16
!pip install -q transformers==4.35.0
!pip install -q datasets==2.14.6
!pip install -q torchaudio==2.1.0
!pip install -q librosa==0.10.1
!pip install -q soundfile==0.12.1
!pip install -q grad-cam==1.4.8
!pip install -q pydub==0.25.1
!pip install -q gTTS==2.4.0
!pip install -q elevenlabs==0.2.27

print("✅ Semua library sudah terinstall!")

ERROR: Could not find a version that satisfies the requirement torchaudio==2.1.0 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torchaudio==2.1.0
✅ Semua library sudah terinstall!


In [4]:
# Sel 2: Import semua
import torch
import torchaudio
import librosa
import soundfile as sf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Cek GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔥 Device: {device}")

In [ ]:
# Sel 3: Download RawNet3 (model anti-spoofing ringan)
!git clone https://github.com/Jungjee/RawNet.git
%cd RawNet/python/RawNet3

# Download checkpoint pretrained
!wget https://github.com/Jungjee/RawNet/releases/download/v1.0/RawNet3_model.pth

# Load model
import sys
sys.path.append('/content/RawNet/python/RawNet3')
from model import RawNet3

# Inisialisasi model
rawnet_model = RawNet3(
    d_args={'nb_samp': 64000, 'first_conv': 1024, 'in_channels': 1,
            'filts': [128, [128,128], [128,256], [256,256]],
            'nb_fc_node': 1024, 'gru_node': 1024, 'nb_gru_layer': 3}
)

# Load pretrained weights
rawnet_model.load_state_dict(torch.load('RawNet3_model.pth', map_location=device))
rawnet_model = rawnet_model.to(device)
rawnet_model.eval()

print("✅ RawNet3 siap dipakai!")

In [ ]:
# Sel 4: Download Wav2Vec2 fine-tuned untuk age estimation
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2Processor

# Model untuk age classification (sudah di-train di VoxCeleb)
model_name = "superb/wav2vec2-base-superb-sid"  # Bisa dipakai untuk age proxy
age_processor = Wav2Vec2Processor.from_pretrained(model_name)
age_model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name)
age_model = age_model.to(device)
age_model.eval()

print("✅ Age estimation model siap!")

In [ ]:
# Sel 5: Generate synthetic voice pakai gTTS (gratis)
from gtts import gTTS
import os

# Buat folder
os.makedirs('synthetic_samples', exist_ok=True)

# Teks Indonesia
texts = [
    "Halo, nama saya Budi, saya berusia dua puluh lima tahun",
    "Selamat pagi, saya seorang mahasiswa dari Jakarta",
    "Perkenalkan, saya bekerja sebagai guru di sekolah dasar"
]

# Generate 10 synthetic samples
for i, text in enumerate(texts):
    tts = gTTS(text=text, lang='id', slow=False)
    tts.save(f'synthetic_samples/synthetic_{i}.wav')

print(f"✅ Generated {len(texts)} synthetic Indonesian voices!")

In [ ]:
# Sel 6: Rekam suara langsung dari mikrofon
from google.colab import files
from IPython.display import HTML, Audio
from google.colab.output import eval_js
from base64 import b64decode
import io

# JavaScript untuk rekam audio
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record_audio(sec=5):
    display(HTML(f'🎤 Rekam selama {sec} detik...'))
    s = eval_js('record(%d)' % (sec*1000))
    b = b64decode(s.split(',')[1])

    # Save
    os.makedirs('real_samples', exist_ok=True)
    with open('real_samples/real_voice.wav', 'wb') as f:
        f.write(b)

    return 'real_samples/real_voice.wav'

# Jalankan ini untuk rekam
print("Klik tombol allow microphone, lalu tunggu 5 detik")
audio_path = record_audio(sec=5)
print(f"✅ Audio tersimpan di: {audio_path}")

# Dengarkan hasil
display(Audio(audio_path))

In [ ]:
# Sel 7: Fungsi untuk deteksi synthetic/real
def predict_synthetic(audio_path, model=rawnet_model):
    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000, duration=4)

    # Pastikan panjang 64000 sample (4 detik @ 16kHz)
    if len(audio) < 64000:
        audio = np.pad(audio, (0, 64000 - len(audio)))
    else:
        audio = audio[:64000]

    # Konversi ke tensor
    audio_tensor = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(device)

    # Prediksi
    with torch.no_grad():
        output = model(audio_tensor)
        # RawNet3 output: score tinggi = bonafide (real), rendah = spoof (synthetic)
        score = output.squeeze().cpu().numpy()

    is_real = score > 0  # threshold 0
    confidence = abs(score)

    return {
        'prediction': 'REAL HUMAN' if is_real else 'SYNTHETIC',
        'confidence': float(confidence),
        'raw_score': float(score)
    }

# Test synthetic
result = predict_synthetic('synthetic_samples/synthetic_0.wav')
print(f"🔍 Synthetic sample → {result}")

# Test real (kalau sudah rekam)
if os.path.exists('real_samples/real_voice.wav'):
    result = predict_synthetic('real_samples/real_voice.wav')
    print(f"🔍 Real sample → {result}")

In [ ]:
# Sel 8: Age estimation (pakai acoustic features)
def estimate_age(audio_path):
    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000)

    # Extract features yang berhubungan dengan usia
    # Pitch (F0) - makin tua cenderung lebih rendah
    pitches, magnitudes = librosa.piptrack(y=audio, sr=sr)
    pitch_mean = np.mean(pitches[pitches > 0]) if np.any(pitches > 0) else 0

    # Jitter (voice instability - makin tua makin tinggi)
    rms = librosa.feature.rms(y=audio)[0]
    jitter = np.std(rms) / (np.mean(rms) + 1e-8)

    # Formant estimate (simplified)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    formant_proxy = np.mean(mfcc[1:4])

    # Rule-based classification (bisa diganti ML model nanti)
    if pitch_mean > 220:  # Hz
        age_group = "ANAK/REMAJA (< 18 tahun)"
    elif pitch_mean > 180:
        age_group = "DEWASA MUDA (18-35 tahun)"
    elif pitch_mean > 140:
        age_group = "DEWASA (35-55 tahun)"
    else:
        age_group = "LANSIA (> 55 tahun)"

    return {
        'age_group': age_group,
        'pitch_hz': float(pitch_mean),
        'voice_stability': float(1/jitter) if jitter > 0 else 100
    }

# Test
age_result = estimate_age('synthetic_samples/synthetic_0.wav')
print(f"👤 Age estimate → {age_result}")

In [ ]:
# Sel 9: Test semua file sekaligus
import glob

def evaluate_all():
    results = []

    # Test synthetic files
    for audio_file in glob.glob('synthetic_samples/*.wav'):
        synth_result = predict_synthetic(audio_file)
        age_result = estimate_age(audio_file)

        results.append({
            'file': audio_file,
            'true_label': 'SYNTHETIC',
            'predicted': synth_result['prediction'],
            'confidence': synth_result['confidence'],
            'age_group': age_result['age_group'],
            'pitch': age_result['pitch_hz']
        })

    # Test real files
    for audio_file in glob.glob('real_samples/*.wav'):
        synth_result = predict_synthetic(audio_file)
        age_result = estimate_age(audio_file)

        results.append({
            'file': audio_file,
            'true_label': 'REAL',
            'predicted': synth_result['prediction'],
            'confidence': synth_result['confidence'],
            'age_group': age_result['age_group'],
            'pitch': age_result['pitch_hz']
        })

    df = pd.DataFrame(results)
    return df

# Jalankan evaluasi
df_results = evaluate_all()
print("\n📊 HASIL EVALUASI:")
print(df_results)

# Hitung akurasi
if len(df_results) > 0:
    accuracy = (df_results['true_label'] == df_results['predicted']).mean() * 100
    print(f"\n✅ Akurasi Deteksi Synthetic: {accuracy:.2f}%")

In [ ]:
# Sel 10: Visualisasi mana bagian audio yang penting untuk klasifikasi
def visualize_audio_importance(audio_path):
    # Load audio
    audio, sr = librosa.load(audio_path, sr=16000)

    # Buat spectrogram
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)

    # Plot
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Spectrogram: {audio_path}')
    plt.tight_layout()
    plt.show()

    # Waveform
    plt.figure(figsize=(12, 3))
    plt.plot(audio)
    plt.title('Waveform')
    plt.xlabel('Sample')
    plt.ylabel('Amplitude')
    plt.tight_layout()
    plt.show()

# Test visualisasi
visualize_audio_importance('synthetic_samples/synthetic_0.wav')

In [ ]:
# Sel 11: Export hasil
df_results.to_csv('hasil_evaluasi.csv', index=False)
print("✅ Hasil disimpan ke hasil_evaluasi.csv")

# Download file
from google.colab import files
files.download('hasil_evaluasi.csv')